Step 0 - venv

In [2]:
# 1. Install python3.10 and venv support
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y
# 2. Create an isolated virtual environment
!python3.10 -m venv /content/venv
# 3. Upgrade pip inside the virtual environment
!/content/venv/bin/python -m pip install --upgrade pip
# 4. Install the required serving pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"
print("Virtual environment ready with vLLM installed!")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,920 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-securi

Step 1 — Cell 1: install + launch the locked model

In [3]:
import subprocess, sys

# Pins mirrored from ../../PINS.md. Keep these two in sync (PINS.md wins).
VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = ["/content/venv/bin/python", "-m", "pip", "install", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [4]:
pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"autoawq=={AUTOAWQ_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)
# NOTE (2026-08-07, verified the hard way on a live T4): do NOT add a
# numpy>=2 pin here - vLLM 0.6.x requires numpy<2 and the install fails
# outright. CELL B as verified 2026-07-27 runs on the numpy vLLM chooses.
print("serving pins installed")

installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* autoawq==0.2.* httpx==0.27.* openai==1.54.*
serving pins installed


In [5]:
import os, signal, subprocess

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
PORT = 8000
SERVER_LOG = "/content/server.log"

# Args as a dict so a lab can override one value without retyping the line.
SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    cmd = ["/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:            # bare flag, e.g. "--enable-auto-tool-choice": None
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    # start_new_session=True puts the server in its own process group so the
    # shutdown cell can kill the whole group, not just the parent pid.
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server(SERVER_ARGS)

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 4910, logging to /content/server.log


In [6]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass  # not up yet
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading "
          "(rerun this cell), OOM at load (lower --gpu-memory-utilization to "
          "0.80), or a bad flag (bf16 on sm75; use --dtype half).")
    return False

healthy = wait_for_health()

server healthy after about 60s: http://localhost:8000/v1/models -> 200


Step 2 — Cell 2: get the harness and the prompts

In [7]:
!python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

[level 1] tok/s=83.5 ttft_p95=0.0847 errors=0
[level 2] tok/s=158.85 ttft_p95=0.0795 errors=0
[level 4] tok/s=275.37 ttft_p95=0.1225 errors=0
[level 8] tok/s=463.05 ttft_p95=0.1718 errors=0
[level 16] tok/s=723.84 ttft_p95=0.2508 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     83.50      0.054      0.085     1.539    20     0
   2    158.85      0.054      0.080     1.616    20     0
   4    275.37      0.065      0.122     1.770    20     0
   8    463.05      0.124      0.172     1.997    20     0
  16    723.84      0.248      0.251     2.401    20     0

wrote bench_report.json (run appended)


Step 3 — Cell 4: find the knee

In [8]:
import json

levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]

for L in levels:
    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"ttft_p95={L['ttft_p95_s']:.3f}  "
        f"lat_p95={L['latency_p95_s']:.3f}  "
        f"errors={L['errors']}"
    )

TARGET_P95_S = 5.0   # your prediction-card SLO

under = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under, key=lambda L: L["concurrency"]) if under else None

print("knee:", knee)

c= 1  tok/s=   83.5  ttft_p95=0.085  lat_p95=1.539  errors=0
c= 2  tok/s=  158.8  ttft_p95=0.080  lat_p95=1.616  errors=0
c= 4  tok/s=  275.4  ttft_p95=0.122  lat_p95=1.770  errors=0
c= 8  tok/s=  463.1  ttft_p95=0.172  lat_p95=1.997  errors=0
c=16  tok/s=  723.8  ttft_p95=0.251  lat_p95=2.401  errors=0
knee: {'concurrency': 16, 'tokens_per_s': 723.84, 'ttft_p50_s': 0.248, 'ttft_p95_s': 0.2508, 'latency_p95_s': 2.4006, 'errors': 0, 'ok': 20, 'wall_s': 2.781}


Step 4 - Cell 5: fill capacity-note.md

In [9]:
import json

with open("knee.json", "w") as f:
    json.dump({
        "target_p95_s": TARGET_P95_S,
        "knee_concurrency": knee["concurrency"] if knee else None
    }, f)

print(open("knee.json").read())

{"target_p95_s": 5.0, "knee_concurrency": 16}


In [13]:
%%writefile capacity-note.md
# Capacity note (team, one page)

## The numbers

- Locked model: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Target p95 end-to-end latency (your SLO today): `5.0 seconds`
- Knee concurrency (highest concurrency whose p95 is still under target):
  `16 (sweep-bounded)`
- Tokens per second at the knee: `723.8 tokens/s`
- Max sustainable request rate at the target p95: `7.2 req/s`

## The limiting family
- Memory-bound is the likely limiting family: decode repeatedly streams model weights from GPU memory, although this sweep did not yet reach the memory-bandwidth ceiling because throughput was still rising at concurrency 16.

## Why the knee, not the peak

- I report the knee at the SLO because it represents the highest concurrency I can sustainably promise while meeting the latency target, whereas peak throughput may occur after latency has become unacceptable.

Overwriting capacity-note.md


In [14]:
!cat capacity-note.md

# Capacity note (team, one page)

## The numbers

- Locked model: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Target p95 end-to-end latency (your SLO today): `5.0 seconds`
- Knee concurrency (highest concurrency whose p95 is still under target):
  `16 (sweep-bounded)`
- Tokens per second at the knee: `723.8 tokens/s`
- Max sustainable request rate at the target p95: `7.2 req/s`

## The limiting family
- Memory-bound is the likely limiting family: decode repeatedly streams model weights from GPU memory, although this sweep did not yet reach the memory-bandwidth ceiling because throughput was still rising at concurrency 16.

## Why the knee, not the peak

- I report the knee at the SLO because it represents the highest concurrency I can sustainably promise while meeting the latency target, whereas peak throughput may occur after latency has become unacceptable.


In [15]:
!cat knee.json

{"target_p95_s": 5.0, "knee_concurrency": 16}

# **Verifiction**

In [16]:
!python verify_w3d5.py

levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS
